In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Fine-tune the Phikon ViT backbone with LoRA adapters using Hugging Face PEFT.

Requirements:
    pip install torch transformers datasets accelerate peft[torch]
"""

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

from datasets import load_dataset
from accelerate import Accelerator

from transformers import (
    AutoFeatureExtractor,
    AutoModelForImageClassification,
    TrainingArguments,
    default_data_collator,
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    TaskType,
)


/home/yuxin/miniconda3/envs/lora/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# 1. Load processor & base model
import os
os.environ["HTTP_PROXY"]  = "http://localhost:7890"
os.environ["HTTPS_PROXY"] = "http://localhost:7890"

os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
MODEL_NAME = "owkin/phikon"
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)


/home/yuxin/miniconda3/envs/lora/lib/python3.11/site-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [3]:
from transformers import AutoConfig, AutoModel
import torch.nn as nn
MODEL_NAME = "owkin/phikon"
# 1) Load backbone (no classifier head)
config = AutoConfig.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained(MODEL_NAME, config=config)

# 2) Define dual‐head classifier
class DualHeadImageClassifier(nn.Module):
    def __init__(self, backbone, hidden_size, num_labels_head1=3, num_labels_head2=3):
        super().__init__()
        self.backbone = backbone
        self.head1 = nn.Linear(hidden_size, num_labels_head1)
        self.head2 = nn.Linear(hidden_size, num_labels_head2)

    def forward(self, pixel_values, labels=None, **kwargs):
        # get the pooled [CLS] token representation
        outputs = self.backbone(pixel_values=pixel_values, return_dict=True)
        if hasattr(outputs, "pooler_output"):
            pooled = outputs.pooler_output
        else:
            # fallback for models without pooler_output
            pooled = outputs.last_hidden_state[:, 0]
        
        logits1 = self.head1(pooled)
        logits2 = self.head2(pooled)

        loss = None
        if labels is not None:
            # assume labels is a dict with 'labels1' and 'labels2'
            loss_fct = nn.CrossEntropyLoss()
            loss1 = loss_fct(logits1, labels["labels1"])
            loss2 = loss_fct(logits2, labels["labels2"])
            loss = loss1 + loss2

        return {"loss": loss, "logits1": logits1, "logits2": logits2}


In [7]:


# 2. Configure and wrap with LoRA
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,  # ← use FEATURE_EXTRACTION, not IMAGE_CLASSIFICATION
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.2,
    target_modules=["query", "value"],
)

# 3) Instantiate your model with two 3‐class heads
model = DualHeadImageClassifier(
    backbone,
    hidden_size=config.hidden_size,
    num_labels_head1=3,
    num_labels_head2=3,
)


In [8]:
model

DualHeadImageClassifier(
  (backbone): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=768, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.2, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=768, bias=False)
                )
                (lora_embedding_A): ParameterDict()
    

In [9]:

# 4) (Optional) Wrap with LoRA adapters exactly as before
model = get_peft_model(model, lora_config)
#model = PeftModel.from_pretrained(model, "phikon-lora")
model.print_trainable_parameters()  # sanity check: should only list LoRA params


trainable params: 294,912 || all params: 86,688,774 || trainable%: 0.3402


/home/yuxin/miniconda3/envs/lora/lib/python3.11/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(


In [5]:
import pandas as pd, torch
import pathlib
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset

class BCaDataset(Dataset):
    def __init__(self, data_root: pathlib.Path, feature_extractor):
        self.data_root = data_root
        self.fe = feature_extractor
        self.images_path = list(data_root.glob('**/*.*'))
        self.images_class = [self._get_type_grade(path) for path in self.images_path]


    def _get_type_grade(self, path):
        type_grade = path.parent.name

        if type_grade == 'normal':
            return 0, 0
        elif type_grade == 'tis-1':
            return 1, 0
        elif type_grade == 'tis-2':
            return 1, 1
        elif type_grade == 'tis-3':
            return 1, 2
        elif type_grade == 'it-1':
            return 2, 0
        elif type_grade == 'it-2':
            return 2, 1
        elif type_grade == 'it-3':
            return 2, 2

    def __len__(self):
        return len(self.images_path)

    def __getitem__(self, idx):
        img = Image.open(self.images_path[idx]).convert("RGB")
        enc = self.fe(images=img, return_tensors="pt")
        pix = enc["pixel_values"].squeeze(0)
        return {
            "pixel_values": pix,
            "labels1": torch.tensor(self.images_class[idx][0], dtype=torch.long),
            "labels2": torch.tensor(self.images_class[idx][1], dtype=torch.long),
        }

train_root = Path('/mnt/hd0/project/bcacad/data/patch-level/suqh/train')
train_dataset = BCaDataset(train_root, feature_extractor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=default_data_collator)

val_root = Path('/mnt/hd0/project/bcacad/data/patch-level/suqh/val')
val_dataset = BCaDataset(val_root, feature_extractor)
eval_loader = DataLoader(val_dataset, batch_size=16, shuffle=True, collate_fn=default_data_collator)


In [9]:

# 4. Prepare training
# accelerator = Accelerator()

# optimizer = AdamW(model.parameters(), lr=1e-5)

# model, optimizer, train_loader, eval_loader = accelerator.prepare(
#     model, optimizer, train_loader, eval_loader
# )

# 5. Training loop
NUM_EPOCHS = 150
for epoch in range(50,NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        outputs = model(
            pixel_values=batch["pixel_values"],
            labels={"labels1": batch["labels1"], "labels2": batch["labels2"]},
        )
        loss = outputs['loss']
        accelerator.backward(loss)
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    correct1, correct2, total = 0, 0, 0

    with torch.no_grad():
        for batch in eval_loader:
            # 1) forward pass (no labels → no loss)
            outputs = model(pixel_values=batch["pixel_values"])
            logits1 = outputs["logits1"]
            logits2 = outputs["logits2"]

            # 2) get predictions
            preds1 = torch.argmax(logits1, dim=-1)
            preds2 = torch.argmax(logits2, dim=-1)

            # 3) accumulate correct counts
            correct1 += (preds1 == batch["labels1"]).sum().item()
            correct2 += (preds2 == batch["labels2"]).sum().item()
            total   += preds1.size(0)

    # 4) compute accuracies
    val_acc1 = correct1 / total
    val_acc2 = correct2 / total

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} — "
        f"Train Loss: {avg_train_loss:.4f} — "
        f"Val Acc (Type): {val_acc1:.4f} — "
        f"Val Acc (Grade): {val_acc2:.4f}"
    )


# 6. Save LoRA adapters
OUTPUT_DIR = "phikon-lora-3"
model.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapters saved to `{OUTPUT_DIR}`")

# 7. Inference example
# base = AutoModelForImageClassification.from_pretrained(MODEL_NAME, num_labels=...)
# lora_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
# lora_model.eval()
# 
# # prepare your image and then:
# inputs = feature_extractor(samples, return_tensors="pt")
# logits = lora_model(**inputs).logits
# preds = logits.argmax(dim=-1)


Epoch 51/150 — Train Loss: 0.0723 — Val Acc (Type): 0.9915 — Val Acc (Grade): 0.9757
Epoch 52/150 — Train Loss: 0.0704 — Val Acc (Type): 0.9910 — Val Acc (Grade): 0.9748
Epoch 53/150 — Train Loss: 0.0688 — Val Acc (Type): 0.9914 — Val Acc (Grade): 0.9746
Epoch 54/150 — Train Loss: 0.0670 — Val Acc (Type): 0.9915 — Val Acc (Grade): 0.9736
Epoch 55/150 — Train Loss: 0.0658 — Val Acc (Type): 0.9914 — Val Acc (Grade): 0.9750
Epoch 56/150 — Train Loss: 0.0638 — Val Acc (Type): 0.9909 — Val Acc (Grade): 0.9769
Epoch 57/150 — Train Loss: 0.0618 — Val Acc (Type): 0.9878 — Val Acc (Grade): 0.9772
Epoch 58/150 — Train Loss: 0.0606 — Val Acc (Type): 0.9913 — Val Acc (Grade): 0.9782
Epoch 59/150 — Train Loss: 0.0583 — Val Acc (Type): 0.9915 — Val Acc (Grade): 0.9770
Epoch 60/150 — Train Loss: 0.0578 — Val Acc (Type): 0.9915 — Val Acc (Grade): 0.9781
Epoch 61/150 — Train Loss: 0.0565 — Val Acc (Type): 0.9922 — Val Acc (Grade): 0.9774
Epoch 62/150 — Train Loss: 0.0545 — Val Acc (Type): 0.9920 — Val 

In [10]:
from tqdm import tqdm
import sys

from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score, f1_score

def evaluate_lora(model, data_loader, device, tasks):
    model.eval()
    y_true = {t: [] for t in tasks}
    y_pred = {t: [] for t in tasks}
    with torch.no_grad():
        for batch in tqdm(data_loader, file=sys.stdout):
            images = batch["pixel_values"].to(device)
            
            label1 = batch["labels1"]
            label2 = batch["labels2"]
            outputs = model(pixel_values=images)
            # type
            if 'type' in tasks:
                pred = outputs['logits1'].argmax(dim=1)
                y_pred['type'].extend(pred.cpu().tolist())
                y_true['type'].extend(label1.cpu().tolist())
            # nonibc
            if 'nonibc' in tasks:
                mask = label1==1
                if mask.any():
                    pred = outputs['logits2'][mask].argmax(dim=1)
                    y_pred['nonibc'].extend(pred.cpu().tolist())
                    y_true['nonibc'].extend(label2[mask].cpu().tolist())
            # ibc
            if 'ibc' in tasks:
                mask = label1==2
                if mask.any():
                    pred = outputs['logits2'][mask].argmax(dim=1)
                    y_pred['ibc'].extend(pred.cpu().tolist())
                    y_true['ibc'].extend(label2[mask].cpu().tolist())
    metrics = {}
    for task in tasks:
        if y_true[task]:
            metrics[task] = {
                'balanced_acc': balanced_accuracy_score(y_true[task], y_pred[task]),
                'kappa': cohen_kappa_score(y_true[task], y_pred[task], weights='quadratic'),
                'f1_macro': f1_score(y_true[task], y_pred[task], average='macro')
            }
        else:
            metrics[task] = {k: None for k in ['balanced_acc','kappa','f1_macro']}
    return metrics

In [11]:
def test_lora(model, data_root, cohort_tasks,device):
    for cohort, tasks in cohort_tasks.items():
        print(f"Evaluating {cohort}...")
        dataset = BCaDataset(data_root / cohort / 'test', feature_extractor)
        loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0, pin_memory=True, collate_fn=default_data_collator)
        metrics = evaluate_lora(model, loader, device, tasks)
        print(f"\n{cohort.upper()}:")
        for t, m in metrics.items():
            print(f"{t}: {m}")

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_root = Path('/mnt/hd0/project/bcacad/data/patch-level')

# Define tasks for each cohort
cohort_tasks = {
    'suqh': ['type', 'nonibc', 'ibc'],
    'qduh': ['type', 'nonibc', 'ibc'],
    'shsu': ['type', 'nonibc', 'ibc'],
    'bracs': ['type'],
    'bcnb': ['ibc'],
    'bach': ['type'],
    'apght': ['ibc'],
    'aggregate': ['type', 'nonibc', 'ibc']
}


test_lora(model, data_root, cohort_tasks,device)


Evaluating suqh...
100%|██████████| 2959/2959 [04:18<00:00, 11.45it/s]

SUQH:
type: {'balanced_acc': 0.9079782068537926, 'kappa': 0.9240534170518273, 'f1_macro': 0.9087276831766342}
nonibc: {'balanced_acc': 0.48920093777416684, 'kappa': 0.4319570340060527, 'f1_macro': 0.479731861151176}
ibc: {'balanced_acc': 0.5790457052373273, 'kappa': 0.4421031782234631, 'f1_macro': 0.5306583873459014}
Evaluating qduh...
100%|██████████| 341/341 [01:07<00:00,  5.05it/s]

QDUH:
type: {'balanced_acc': 0.6494917119467306, 'kappa': 0.6066747579102685, 'f1_macro': 0.5649810884059772}
nonibc: {'balanced_acc': 0.4056801214225616, 'kappa': 0.27191712375365396, 'f1_macro': 0.38841944487110175}
ibc: {'balanced_acc': 0.5041846297958827, 'kappa': 0.36891687761460845, 'f1_macro': 0.4864626921070143}
Evaluating shsu...
100%|██████████| 285/285 [00:57<00:00,  4.94it/s]

SHSU:
type: {'balanced_acc': 0.8276058677779029, 'kappa': 0.6932516140237759, 'f1_macro': 0.5690587400922588}
nonibc: {'balanced_acc': 0.4519063722

/home/yuxin/miniconda3/envs/lora/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2776: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


100%|██████████| 631/631 [02:59<00:00,  3.52it/s]

BRACS:
type: {'balanced_acc': 0.8402222664759719, 'kappa': 0.7126987574952521, 'f1_macro': 0.7884598222898213}
Evaluating bcnb...
100%|██████████| 606/606 [02:43<00:00,  3.72it/s]

BCNB:
ibc: {'balanced_acc': 0.4718488108256593, 'kappa': 0.12257652278406961, 'f1_macro': 0.3210647938199162}
Evaluating bach...
100%|██████████| 75/75 [00:20<00:00,  3.64it/s]

BACH:
type: {'balanced_acc': 0.8008333333333333, 'kappa': 0.7749648382559775, 'f1_macro': 0.7925224490300146}
Evaluating apght...
100%|██████████| 19/19 [00:02<00:00,  6.77it/s]

APGHT:
ibc: {'balanced_acc': 0.4627990664274337, 'kappa': 0.33697347893915763, 'f1_macro': 0.45879912751823615}
Evaluating aggregate...
100%|██████████| 4913/4913 [08:18<00:00,  9.86it/s]

AGGREGATE:
type: {'balanced_acc': 0.8352643194441939, 'kappa': 0.7283554087286541, 'f1_macro': 0.8117670010779118}
nonibc: {'balanced_acc': 0.4921464696011945, 'kappa': 0.3308516067114773, 'f1_macro': 0.4941661659672824}
i